In [1]:
import glob, os
import pandas as pd

path = glob.glob(os.path.expanduser("~/openpom/**/*4983*.csv"), recursive=True)[0]
df = pd.read_csv(path)
print(path)
print("rows, columns:", df.shape)
print(df[["fruity", "smoky"]].sum())
df.head(3)

/home/sachinkukkar/openpom/openpom/data/curated_datasets/curated_GS_LF_merged_4983.csv
rows, columns: (4983, 140)
fruity    1902
smoky       72
dtype: int64


,nonStereoSMILES,descriptors,alcoholic,aldehydic,alliaceous,almond,amber,animal,anisic,apple,...,tropical,vanilla,vegetable,vetiver,violet,warm,waxy,weedy,winey,woody
0,CC(O)CN,fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CCC(=O)C(=O)O,fatty;lactonic;sweet;caramellic;creamy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,O=C(O)CCc1ccccc1,rose;floral;fatty;sweet;musk;cinnamon;balsamic,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [2]:
import os, shutil, inspect
import deepchem as dc
from openpom.feat.graph_featurizer import GraphFeaturizer
from openpom.utils.data_utils import IterativeStratifiedSplitter

# 1. The 138 smell words are every column after the first two
TASKS = list(df.columns[2:])
print("smell words:", len(TASKS))

# 2. Turn each SMILES into a graph and save it to disk
DATA = os.path.expanduser("~/ai-nose/data")
shutil.rmtree(DATA, ignore_errors=True)          # start fresh if you re-run this cell
for d in ["all", "train", "valid", "test"]:
    os.makedirs(f"{DATA}/{d}", exist_ok=True)

loader = dc.data.CSVLoader(tasks=TASKS, feature_field="nonStereoSMILES", featurizer=GraphFeaturizer())
dataset = loader.create_dataset(inputs=[path], data_dir=f"{DATA}/all")
print("molecules turned into graphs:", len(dataset))

# 3. Split 80/10/10 with iterative stratification (the paper's method)
sig = inspect.signature(IterativeStratifiedSplitter)
print("splitter options:", sig)
splitter = IterativeStratifiedSplitter(order=2) if "order" in sig.parameters else IterativeStratifiedSplitter()
train, valid, test = splitter.train_valid_test_split(
    dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1,
    train_dir=f"{DATA}/train", valid_dir=f"{DATA}/valid", test_dir=f"{DATA}/test")
print("train / valid / test:", len(train), len(valid), len(test))

# 4. Check: does every smell word appear at least once in the test set?
missing = int((test.y.sum(axis=0) == 0).sum())
print("smell words missing from the test set:", missing)

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading material featurizer, missing a dependency. No module named 'ase'
/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading some PyTorch models, missing a dependency. No module

smell words: 138
molecules turned into graphs: 4983
splitter options: (order: int = 2) -> None
train / valid / test: 3978 510 495
smell words missing from the test set: 0


In [3]:
import os, time
import deepchem as dc
from openpom.feat.graph_featurizer import GraphConvConstants
from openpom.utils.data_utils import get_class_imbalance_ratio
from openpom.models.mpnn_pom import MPNNPOMModel

# Load the split we saved in Step 7
DATA = os.path.expanduser("~/ai-nose/data")
train = dc.data.DiskDataset(f"{DATA}/train")
valid = dc.data.DiskDataset(f"{DATA}/valid")
test  = dc.data.DiskDataset(f"{DATA}/test")
print("train / valid / test:", len(train), len(valid), len(test))
print("smell words missing from valid:", int((valid.y.sum(axis=0) == 0).sum()))

# Give rare smell words more weight, so the model doesn't ignore them
train_ratios = get_class_imbalance_ratio(train)

# Learning rate: starts at 0.001 and halves every 20 epochs
learning_rate = dc.models.optimizers.ExponentialDecay(
    initial_rate=0.001, decay_rate=0.5, decay_steps=32*20, staircase=True)

model = MPNNPOMModel(
    n_tasks=len(train.tasks), batch_size=128, learning_rate=learning_rate,
    class_imbalance_ratio=train_ratios, loss_aggr_type="sum",
    node_out_feats=100, edge_hidden_feats=75, edge_out_feats=100,
    num_step_message_passing=5, mpnn_residual=True, message_aggregator_type="sum",
    mode="classification", number_atom_features=GraphConvConstants.ATOM_FDIM,
    number_bond_features=GraphConvConstants.BOND_FDIM, n_classes=1,
    readout_type="set2set", num_step_set2set=3, num_layer_set2set=2,
    ffn_hidden_list=[392, 392], ffn_embeddings=256, ffn_activation="relu",
    ffn_dropout_p=0.12, ffn_dropout_at_input_no_act=False, weight_decay=1e-5,
    self_loop=False, optimizer_name="adam", log_frequency=32,
    model_dir=os.path.expanduser("~/ai-nose/models/first_run"), device_name="cuda")

metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    loss = model.fit(train, nb_epoch=1, max_checkpoints_to_keep=1,
                     deterministic=False, restore=epoch > 1)
    secs = time.time() - t0
    tr = model.evaluate(train, [metric])["roc_auc_score"]
    va = model.evaluate(valid, [metric])["roc_auc_score"]
    print(f"epoch {epoch}/{EPOCHS} | {secs:.0f}s | loss {loss:.3f} | train AUROC {tr:.3f} | valid AUROC {va:.3f}")

model.save_checkpoint()
print("TEST AUROC:", round(model.evaluate(test, [metric])["roc_auc_score"], 3))

train / valid / test: 3978 510 495
smell words missing from valid: 0
epoch 1/5 | 17s | loss 3.990 | train AUROC 0.613 | valid AUROC 0.605


/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data

epoch 2/5 | 9s | loss 3.301 | train AUROC 0.696 | valid AUROC 0.697


/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data

epoch 3/5 | 4s | loss 3.086 | train AUROC 0.778 | valid AUROC 0.772


/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data

epoch 4/5 | 8s | loss 2.974 | train AUROC 0.800 | valid AUROC 0.795


/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data

epoch 5/5 | 4s | loss 2.901 | train AUROC 0.821 | valid AUROC 0.820
TEST AUROC: 0.812


In [4]:
import warnings, pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)   # hide the torch.load notice

def make_model(model_dir):
    lr = dc.models.optimizers.ExponentialDecay(
        initial_rate=0.001, decay_rate=0.5, decay_steps=32*20, staircase=True)
    return MPNNPOMModel(
        n_tasks=len(train.tasks), batch_size=128, learning_rate=lr,
        class_imbalance_ratio=train_ratios, loss_aggr_type="sum",
        node_out_feats=100, edge_hidden_feats=75, edge_out_feats=100,
        num_step_message_passing=5, mpnn_residual=True, message_aggregator_type="sum",
        mode="classification", number_atom_features=GraphConvConstants.ATOM_FDIM,
        number_bond_features=GraphConvConstants.BOND_FDIM, n_classes=1,
        readout_type="set2set", num_step_set2set=3, num_layer_set2set=2,
        ffn_hidden_list=[392, 392], ffn_embeddings=256, ffn_activation="relu",
        ffn_dropout_p=0.12, ffn_dropout_at_input_no_act=False, weight_decay=1e-5,
        self_loop=False, optimizer_name="adam", log_frequency=32,
        model_dir=model_dir, device_name="cuda")

RUN = os.path.expanduser("~/ai-nose/models/full_run")
BEST = RUN + "/best"
os.makedirs(BEST, exist_ok=True)
model = make_model(RUN)

EPOCHS, CHECK_EVERY = 150, 5
best_valid, best_epoch, history = 0.0, 0, []
start = time.time()
for epoch in range(1, EPOCHS + 1):
    loss = model.fit(train, nb_epoch=1, max_checkpoints_to_keep=1,
                     deterministic=False, restore=epoch > 1)
    if epoch % CHECK_EVERY == 0:
        tr = model.evaluate(train, [metric])["roc_auc_score"]
        va = model.evaluate(valid, [metric])["roc_auc_score"]
        history.append({"epoch": epoch, "loss": loss, "train_auroc": tr, "valid_auroc": va})
        flag = ""
        if va > best_valid:
            best_valid, best_epoch = va, epoch
            model.save_checkpoint(max_checkpoints_to_keep=1, model_dir=BEST)
            flag = "  <- best so far, saved"
        mins = (time.time() - start) / 60
        print(f"epoch {epoch:3d} | {mins:5.1f} min | loss {loss:.3f} | train {tr:.3f} | valid {va:.3f}{flag}")

pd.DataFrame(history).to_csv(RUN + "/history.csv", index=False)
model.restore(model_dir=BEST)
test_auroc = model.evaluate(test, [metric])["roc_auc_score"]
print(f"\nBest valid AUROC {best_valid:.3f} at epoch {best_epoch}")
print(f"TEST AUROC (best model): {test_auroc:.3f}")

epoch   5 |   0.7 min | loss 2.853 | train 0.825 | valid 0.811  <- best so far, saved
epoch  10 |   1.3 min | loss 2.633 | train 0.861 | valid 0.844  <- best so far, saved
epoch  15 |   2.1 min | loss 2.509 | train 0.888 | valid 0.862  <- best so far, saved
epoch  20 |   3.2 min | loss 2.405 | train 0.903 | valid 0.866  <- best so far, saved
epoch  25 |   4.3 min | loss 2.258 | train 0.915 | valid 0.870  <- best so far, saved
epoch  30 |   5.0 min | loss 2.203 | train 0.921 | valid 0.874  <- best so far, saved
epoch  35 |   6.1 min | loss 2.134 | train 0.922 | valid 0.871
epoch  40 |   6.7 min | loss 2.065 | train 0.934 | valid 0.882  <- best so far, saved
epoch  45 |   7.3 min | loss 1.931 | train 0.940 | valid 0.881
epoch  50 |   7.9 min | loss 1.885 | train 0.944 | valid 0.878
epoch  55 |   8.5 min | loss 1.825 | train 0.947 | valid 0.878
epoch  60 |   9.0 min | loss 1.783 | train 0.950 | valid 0.879
epoch  65 |   9.8 min | loss 1.698 | train 0.953 | valid 0.879
epoch  70 |  10.4 mi